# E9 GNN Navigation

Author: Arush Arora

## Introduction
This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the shortest-distance paths of the graph before expecting it to serve the LLM with **multiplicative** GREPs for navigation tasks, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

## The R-PEARL GNN
The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

### Graph Convolutional Network (GNN)
The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

### Random Graph Positional Encodings (R-PEARL)
The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $$\mathbf{Q} \in \mathbb{R}^{M \times N} \qquad \mathbf{Q} \sim \mathcal{N}(0, \mathbf{I}) \qquad \mathbf{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\mathbf{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\mathbf{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\mathbf{P}$:$$\mathbf{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)};\, \mathbf{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \mathbf{S}^k\mathbf{q}^{(m)} {\mathbf{H}}_k\bigg)$$ $$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \mathbf{P}^{(m)}$$

## Sparse Graph Transformer
The Sparse Graph Transformer (hereafter named Graph Transformer or GT) follows the same architecture as that of a normal transformer, albeit that the attention mecahnism is modified to scope only over the $k$-hop neighborhood of the query node. The mathematical equations below express the functionality of the Graph Transformer:
$$\mathbf{X}_L = \Phi\Big(\mathbf{X}_0 + \mathbb{\hat{E}}_{\mathbf{q \sim \mathcal{N}(0,\, \mathbf{I})}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}\right]^\top_{\begin{subarray}{l}t \in [N] \\[2.5pt] U = \mathcal{N}^{\le k}(t)\end{subarray}}$$
$$\mathbf{Y}^{(h)}_l = \left(\mathbf{W}_o\right)^\top_l \mathbf{V}_l \mathbf{X}_{l - 1} \left(\mathbf{A}^{(h)}_l\right)^\top$$
$$\mathbf{X}_l = \sigma\bigg(\sum_{h = 1}^H \mathbf{Y}^{(h)}_l\bigg)$$

## Transformer
The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$$\mathbf{Z}_{1:t}^{(L)} = \operatorname{Trf}\Big(\mathbf{X}_{1:t}, {\mathcal{T}}_l\Big) \qquad {\mathcal{T}}_l = \begin{bmatrix}
\mathbf{Q}_l & \mathbf{K}_l & \mathbf{V}_l & \left(\mathbf{W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big(\mathbf{Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

## Graph-Augmented LLM
The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \mathbf{S}, \mathcal{H}\Big)$$

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$${\mathbf{Z}}_{1:t}^{(L)} = \operatorname{Trf}\Big({\mathbf{X}}_{1:t}; \, \cdot \,\Big)$$

In [ ]:
# Import modules.
import copy
import torch
import random
import wandb
import sympy as sp
import networkx as nx

from torch import nn
from torch_geometric.data import Data
from torch.nn.utils import clip_grad_norm_
from itertools import product, combinations
from torch_geometric.utils import to_networkx
from torch.optim.lr_scheduler import ReduceLROnPlateau

from prism.models.r_pearl import RandomGNNPositionalEncodings
from prism.models.gt import GraphTransformer
from prism.eval import evaluate
from prism.data import data

In [ ]:
# Weights & Biases setup. Mirrors prism.training.train_v3._setup_wandb (project /
# name / tags / group + full-config logging), adapted for this notebook's hand-written
# train loops. Each training stage gets its own run, grouped/tagged by GNN type so the
# R-PEARL and GT variants of the same stage line up on one W&B dashboard. The helpers
# introspect the live optimizer / scheduler / loss objects so EVERY hyperparameter is
# logged without hand-maintaining a list.
WANDB_PROJECT = 'e9-gnn-navigation'


def optimizer_hparams(optimizer):
    """Every optimizer setting: class name, shared defaults, and per-param-group values
    (LRs, betas, eps, weight_decay, ...) with the parameter tensors stripped out."""
    return {
        'optimizer': type(optimizer).__name__,
        'optimizer_defaults': dict(optimizer.defaults),
        'param_groups': [
            {k: v for k, v in g.items() if k != 'params'}
            for g in optimizer.param_groups
        ],
    }


def scheduler_hparams(scheduler):
    """Every LR-scheduler setting (or {'scheduler': None} when unused)."""
    if scheduler is None:
        return {'scheduler': None}
    keys = ('mode', 'factor', 'patience', 'threshold', 'threshold_mode',
            'cooldown', 'min_lrs', 'eps')
    return {
        'scheduler': type(scheduler).__name__,
        **{k: getattr(scheduler, k) for k in keys if hasattr(scheduler, k)},
    }


def loss_hparams(loss_fn):
    """Loss class, reduction, and pos_weight (resolved to plain Python)."""
    out = {'loss_fn': type(loss_fn).__name__,
           'reduction': getattr(loss_fn, 'reduction', None)}
    pos_weight = getattr(loss_fn, 'pos_weight', None)
    if pos_weight is not None:
        out['pos_weight'] = (pos_weight.detach().cpu().tolist()
                             if torch.is_tensor(pos_weight) else pos_weight)
    return out


def init_wandb(stage, hparams):
    """Start a W&B run for a training `stage` ('edge_detection' / 'path_navigation').

    Logs the FULL run config: the GNN construction kwargs (`model_hparams`, set in the
    GNN-instantiation cell) plus every optimizer / scheduler / loss / batching
    hyperparameter the caller assembles in `hparams`. `model_type` selects R-PEARL vs
    GT and drives the run name / tag / group. Returns the run; `reinit=True` so
    successive stages in one notebook session each open a fresh run.
    """
    return wandb.init(
        project=WANDB_PROJECT,
        name=f'{stage}_{model_type}',
        tags=[stage, model_type],
        group=model_type,
        config={'model_type': model_type, 'stage': stage,
                'model': model_hparams, **hparams},
        reinit=True,
    )

In [2]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 0, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [3]:
# Standard options.
eval_path = '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
device = 'cuda'

In [4]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = data.load_samples_by_graph(eval_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [random.choice(eval_data)]}

In [5]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_016.html


## Experiments

### §1 Pretraining a GNN to Classify Edge Existence
We first hope to optimize a GNN (R-PEARL or Graph Transformer) to classify whether an edge exists in the graph or not. Such a model will serve as a backbone pretrained model for fine-tuning on reporting shortest paths. The equations to represent this procedure are below:
$$\mathbf{H} = \mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, \cdot\,)\big]$$
$$\mathbf{\hat{y}}_{ij} = \text{MLP}\big[\mathbf{h}_i\ \Vert\ \mathbf{h}_j\ \Vert\ \mathbf{h}_i \odot \mathbf{h}_j\ \Vert\ |\mathbf{h}_i - \mathbf{h}_j\|\big] \in [0, 1]$$

In [ ]:
# Instantiate a GNN
model_type = 'gt'
if model_type == 'gt':
    model_hparams = dict(
        num_layers=3,
        pe_hidden_channels=256,
        pe_num_layers=5,
        d_model=1024,
        heads=8,
        num_samples=320,
        dropout=0.1,
        k_pe=3,
        k_gt=2,
        eps=1e-6,
        use_layer_norm=True,
    )
    gnn = GraphTransformer(**model_hparams)
else:
    model_hparams = dict(
        pe_hidden_channels=256,
        pe_num_layers=5,
        d_model=1024,
        num_samples=320,
        dropout=0.1,
        k=3,
        eps=1e-6,
        use_layer_norm=True,
    )
    gnn = RandomGNNPositionalEncodings(**model_hparams)
gnn.out_features = gnn.d_model

In [7]:
from typing import Union


# Define a class for edge detection and instantiate it.
class GNNEdgeDetector(nn.Module):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNEdgeDetector, self).__init__()
        shape = gnn.out_features
        self.classifier = nn.Sequential(
            nn.Linear(4 * shape, shape),
            nn.LeakyReLU(),
            nn.Linear(shape, 1)
        )
        self.graph = Data(
            x=torch.empty((0, 0), dtype=torch.float),
            edge_index=torch.empty((2, 0), dtype=torch.long)
        )
        self.cached_pe = torch.zeros(size=(1, shape))
        self.gnn = gnn

    def forward(self, graph: Data, node1: int, node2: int):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        hi, hj = self.cached_pe[node1], self.cached_pe[node2]
        return self.classifier(torch.cat((hi, hj, hi * hj, abs(hi - hj)), dim=0))
    
    def invalidate_cache(self):
        self.graph = None
        self.cached_pe = None


# Instantiate the class.
detector = GNNEdgeDetector(gnn)

#### Numeric Visualizations with SymPy
Using the `render_matrix()` function defined at the very beginning of this notebook, we first explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the eigenbasis of the graph adjacency given a scene graph PyTorch `Data` object.

In [8]:
# Prepare a graph from the data to be used in the GNN.
from torch_geometric.utils import to_dense_adj, to_networkx
from prism.data import utils
import numpy as np
import sympy as sp

# Prepare the graph for rendition.
ex_graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
adj = to_dense_adj(ex_graph.edge_index).squeeze().cuda()
ex_graph.edge_index = ex_graph.edge_index.to(device)
ex_graph.x = ex_graph.x.to(device)

# Show the adjacency matrix of the graph.
render_matrix(adj)

Matrix([
[  0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0],
[  0,   0,   0,   0,   0, 1.0, 1.0, 1.0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0],
[  0,   0,   0,   0, 1.0,   0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0, 1.0],
[  0,   0,   0,   0, 1.0, 1.0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0

In [9]:
# Feed the matrix to the GNN.
out = gnn(ex_graph).to(device)
_, _, V = torch.pca_lowrank(out, q=10, center=True)
out = out - out.mean(dim=0)
render_matrix(out @ V, 3)

Matrix([
[   9.48, -0.264,    1.9,   3.16,    0.86,   0.14,   0.232,  -2.72,  -1.89,   1.86],
[   6.02, -0.784,   1.71, -0.636,   -2.79,   -3.8,   -1.99,    2.9, -0.248,   1.11],
[    3.7,  -4.41,   3.92, -0.798,    2.13,  -1.72,  -0.162,   1.02,   1.29,   1.92],
[   6.41,   1.95,  -1.75,  -1.72,   -1.59,    1.2,    2.05, -0.697,   1.72,  -1.12],
[  0.476,   2.13,  -2.92,   1.15,  -0.836, -0.393,  -0.994,   3.33, -0.338, -0.905],
[  -3.22,    3.6,   2.78,  0.559,   -0.86,  -2.44,   0.699,   1.05,    3.3,  0.638],
[    2.8,  -2.17,  0.248,  -3.11,    1.71,   4.06,    -1.6,  0.291,  0.358,  -2.92],
[  -3.84,   2.03,   1.66,   2.65,   -1.82, 0.0474,    1.69, -0.291,   1.45,  -1.33],
[ -0.344, -0.365,  0.279,   4.79, -0.0163, -0.211,   -1.09,  -3.02,  0.551,  -1.02],
[ -0.347,  -1.45,   1.96,   2.39,   -2.09,   1.98,    2.44,  0.133,    1.8,  -3.73],
[  0.325,    2.8,   3.29, -0.251,   -0.31,   3.94,   0.134, -0.894,  -2.24,   2.37],
[  -4.22,  -1.61,   1.01,  -1.71,    1.33, -0.593,    2.

In [10]:
# Test out the Detector.
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
out = detector(ex_graph, node1, node2).to(device)
render_matrix(torch.sigmoid(out), 3)

Matrix([[0.428]])

#### Pre-Training of GNN on Edge Incidence
Next, we actually preprocess and train the GNN using the steps defined above.

In [11]:
# Import Modules.
from torch_geometric.loader import DataLoader

# Configure the training and test datasets.
train_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)
test_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
)

# Configure the validation dataset.
train_prop = 0.8
train_num = len(train_dataset)
train_keys = random.sample(list(train_dataset.keys()), k=int(train_num * train_prop))
val_dataset = {k: v for k, v in train_dataset.items() if k not in train_keys}
train_dataset = {k: v for k, v in train_dataset.items() if k in train_keys}

# Add edge existence tuples for edge existence.
def generate_data_edges(dataset):
    graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in dataset.items()]
    for graph in graphs:
        graph.x = graph.x.to(device)
        graph.edge_index = graph.edge_index.to(device)
        combs = torch.tensor(
            [[u, v] for u in range(graph.num_nodes) for v in range(u + 1, graph.num_nodes)],
            device=device
        ).T
        existence = torch.tensor(
            [combs[:, i].tolist() in graph.edge_index.T.tolist() for i in range(combs.shape[1])],
            device=device
        )
        graph.exclusion = combs[:, ~existence].to(device)
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones(size=(graph.edge_index.shape[1],)), torch.zeros(size=(indices.shape[0],))), 
            dim=0
        ).to(device)
    
    return graphs


def reshuffle(graphs):
    for graph in graphs:
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones(size=(graph.edge_index.shape[1],)), torch.zeros(size=(indices.shape[0],))), 
            dim=0
        ).to(device)
    
    return graphs


train_graphs_edges = generate_data_edges(train_dataset)
val_graphs_edges = generate_data_edges(val_dataset)
test_graphs_edges = generate_data_edges(test_dataset)

In [ ]:
# Train the GNN to reconstruct the eigenvectors of the graph adjacency.
batch_size = 4
val_freq = 5
epochs = 150
es_patience = 5

def test_loop(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    num_samples = len(dataloader)
    test_loss, correct = 0, 0
    tp = fp = fn = tn = 0

    with torch.no_grad():
        for graph in dataloader.dataset:
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            test_loss += loss_fn(preds, graph.edges_y).item()
            true = graph.edges_y.bool()
            pred = preds > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct += ((preds.sigmoid() > 0.5).float() == graph.edges_y).float().mean().item()

    test_loss /= size
    correct /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, F1: {f1:.3f} | P: {precision:.3f} "
          f"| R: {recall:.3f} | Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/accuracy': correct,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc,
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss


def train_loop(train_dataloader, val_dataloader, test_dataloader, model,
               loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    val_loss: float = 0
    best_val, best_state, bad_runs = float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('edge_detection', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq}\n=============")
            val_loss = test_loop(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss)
            if val_loss < best_val - 1e-3:
                best_val, bad_runs = val_loss, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model.train()
        
        print(f"=============\nEpoch #{i}\n=============")
        optimizer.zero_grad()
        reshuffle(train_dataloader.dataset)
        for j, graph in enumerate(train_dataloader.dataset):
            # Compute prediction and loss.
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            loss = loss_fn(preds, graph.edges_y)

            # Backpropagation.
            (loss / batch_size).backward()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                loss, current = loss.item(), j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)
    
    # Test the finished model.
    test_loop(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


loss_fn = nn.BCEWithLogitsLoss()
train_dataloader = DataLoader(train_graphs_edges, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs_edges, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs_edges, batch_size=batch_size)
optimizer = torch.optim.AdamW([
    {'params': detector.gnn.parameters(), 'lr': 3e-5},
    {'params': detector.classifier.parameters(), 'lr': 3e-4},
], betas=(0.9, 0.95), weight_decay=0.05)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
# train_loop(train_dataloader, val_dataloader, test_dataloader, detector,
#            loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)

In [13]:
# torch.save(detector, '../outputs/e9_multistage_training/edge_detector_final_2.pt')
# torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/edge_detector_{model_type}_final.pt')
gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/edge_detector_gt_final.pt'))
detector = torch.load('../outputs/e9_multistage_training/edge_detector_final_2.pt', weights_only=False)

#### Evaluation of Pre-Trained GNN on Edge Incidence
We now test the trained model on the evaluation dataset. First, we we will render the output for clarity.

In [14]:
# Test out the Detector.
detector.to(device)
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
out = detector(ex_graph, node1, node2)
render_matrix(torch.sigmoid(out), 3)

Matrix([[4.49e-8]])

In [ ]:
# Evaluate the GNN on its reconstruction of eigenvectors of test graph adjacencies.
test_loop(test_dataloader, detector, loss_fn)

Test Error: 
 Accuracy: 92.5%, F1: 0.930 | P: 0.872 | R: 0.996 | Bal Acc: 92.5% | Avg loss: 0.228276 



0.22827626392245293

## Experiments

### §1 Fine-tuning the GNN to Classify Shortest-Path Node Inclusion
We now wish to optimize the pre-trained GNN (R-PEARL or Graph Transformer) to classify which nodes reside on the shortest path between two given nodes in the graph. Such a model will serve as the backbone for the multi-stage training loop featured in E9 Multistage Training of the GREP-PRISM project. The equations to represent this procedure are below:
$$\mathbf{H} = \mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, \cdot\,)\big]$$
$$\mathbf{\hat{y}}_{ij} = \text{MLP}\big[\mathbf{h}_i\ ||\ \mathbf{h}_j\big] \in [0, 1]^N$$

In [ ]:
from typing import Union


# Define a class for edge detection and instantiate it.
class GNNShortestPathNavigator(GNNEdgeDetector):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super().__init__(gnn)
        shape = gnn.out_features
        self.classifier = nn.Sequential(
            nn.Linear(5 * shape, shape),
            nn.LeakyReLU(),
            nn.Linear(shape, 1)
        )

    def forward(self, graph: Data, node1: int, node2: int):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        pe = self.cached_pe
        hi, hj = pe[node1], pe[node2]
        
        features = torch.cat(
            (pe, hi.expand_as(pe), hj.expand_as(pe), pe * hi, pe * hj), dim=1
        )
        return self.classifier(features)


# Instantiate the class.
navigator = GNNShortestPathNavigator(gnn)

In [ ]:
# Test out the Navigator.
navigator.to(device)
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
out = navigator(ex_graph, node1, node2)
render_matrix(torch.sigmoid(out), 3)

#### Fine-Tuning of GNN on Shortest-Path Node Inclusion
Finally, we preprocess and train the GNN using the steps defined above.

In [ ]:
# Add edge existence tuples for edge existence.
def generate_data_paths(dataset):
    graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in dataset.items()]
    for graph in graphs:
        N = graph.num_nodes
        graph.x = graph.x.to(device)
        graph.edge_index = graph.edge_index.to(device)
        graph.paths = torch.zeros(size=(N, N, N)).to(device)
        graph.dist = torch.full((N, N), float('inf')).to(device)
        g = to_networkx(graph, to_undirected=True, edge_attrs=['distance_m'])
        lengths = dict(nx.all_pairs_dijkstra_path_length(g, weight='distance_m'))
        for u in range(N):
            for v in lengths[u]:
                graph.dist[u, v] = lengths[u][v]
                for p in nx.all_shortest_paths(g, u, v, weight='distance_m'):
                    graph.paths[u, v, p] = 1
                    graph.paths[v, u, p] = 1
    
    return graphs

train_graphs_paths = generate_data_paths(train_dataset)
val_graphs_paths = generate_data_paths(val_dataset)
test_graphs_paths = generate_data_paths(test_dataset)

In [ ]:
# Train the GNN to reconstruct the eigenvectors of the graph adjacency.
batch_size = 4
val_freq = 5
epochs = 200
es_patience = 5
detour_bce = False

def test_loop(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    test_loss = tp = fp = fn = tn = 0

    with torch.no_grad():
        for graph in dataloader.dataset:
            for u in range(graph.num_nodes):
                preds = torch.stack(
                    [model(graph, u, v)for v in range(graph.num_nodes)]
                ).squeeze(-1).to(device)
                test_loss += loss_fn(preds, graph.paths[u]).mean().item() / graph.num_nodes
                true = graph.paths[u].bool()
                pred = preds > 0
                tp += (pred & true).sum().item()
                fp += (pred & ~true).sum().item()
                fn += (~pred & true).sum().item()
                tn += (~pred & ~true).sum().item()

    test_loss /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error: \n F1: {f1:.3f} | P: {precision:.3f} | R: {recall:.3f} | "
          f"Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc,
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, f1


def train_loop(train_dataloader, val_dataloader, test_dataloader, model, 
               loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    f1: float = 0
    best_f1, best_state, bad_runs = -float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('path_navigation', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        'detour_bce': detour_bce,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq}\n=============")
            _, f1 = test_loop(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(f1)
            if f1 > best_f1 + 1e-2:
                best_f1, bad_runs = f1, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best F1 {best_f1:>8f})")
                    break
            model.train()
        
        print(f"=============\nEpoch #{i}\n=============")
        optimizer.zero_grad()
        for j, graph in enumerate(train_dataloader.dataset):
            # Compute path metrics for Detour-BCE loss.
            N = graph.num_nodes
            if detour_bce:
                D = graph.dist
                diam = D[torch.isfinite(D)].max()
                delta = (D[:, None, :] + D.transpose(0, 1)[None, :, :] - D[:, :, None]).clamp_min(0)
                delta_norm = (delta / diam).nan_to_num(0.0)
                reachable = torch.isfinite(D)

            # Compute prediction and loss.
            loss = 0
            for u in range(N):
                preds = torch.stack(
                    [model(graph, u, v) for v in range(N)]
                ).squeeze(-1).to(device)

                # Compute Detour-BCE loss.
                raw_loss = loss_fn(preds, graph.paths[u])
                if detour_bce:
                    neg_weights = 1.0 + delta_norm[u]
                    weights = torch.where(
                        graph.paths[u].bool(), torch.ones_like(neg_weights), neg_weights
                    )
                    mask = reachable[u].unsqueeze(-1).float()
                loss = loss + raw_loss / graph.num_nodes

            # Backpropagation.
            (loss / batch_size).backward()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                loss, current = loss.item() / N, j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)

    # Test the finished model.
    test_loop(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# Establish path-vector (sparsity-) sensitive BCE Logit loss.
positive = sum(g.paths.sum() for g in train_graphs_paths)
pos_weight = (sum(g.paths.numel() for g in train_graphs_paths) - positive) / positive
pos_weight **= 0.5
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device), reduction='mean')

train_dataloader = DataLoader(train_graphs_paths, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs_paths, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs_paths, batch_size=batch_size)

optimizer = torch.optim.AdamW([
    {'params': navigator.gnn.parameters(), 'lr': 3e-5},
    {'params': navigator.classifier.parameters(), 'lr': 3e-4},
], betas=(0.9, 0.95), weight_decay=0.05)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
train_loop(train_dataloader, val_dataloader, test_dataloader, navigator,
           loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)

In [ ]:
# torch.save(detector, '../outputs/e9_multistage_training/path_navigator_final_2.pt')
# torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/path_navigator_{model_type}_final.pt')
gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/path_navigator_gt_final.pt'))
navigator = torch.load('../outputs/e9_multistage_training/path_navigator_final_2.pt', weights_only=False)

#### Evaluation of Pre-Trained GNN on Edge Incidence
We thus test the fine-tuned model on the evaluation dataset. First, we we will render the output for clarity.

In [ ]:
# Test out the Navigator.
navigator.to(device)
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
out = navigator(ex_graph, node1, node2)
render_matrix(torch.sigmoid(out), 3)

In [ ]:
# Evaluate the GNN on its reconstruction of eigenvectors of test graph adjacencies.
test_loop(test_dataloader, navigator, loss_fn)